## OpenAI 호환 API와 MLflow 살펴보기

학생과 교수를 위한 AI 어시스턴트라는 RDU의 아이디어가 구체화되기 시작했습니다. 더 복잡한 사용 사례로 넘어가기 전에, 내부에서 어떤 일이 일어나고 있는지 살펴보겠습니다.

이 노트북에서는 현재 Canopy 애플리케이션의 두 가지 핵심 구성 요소를 살펴봅니다.

- **OpenAI 호환 클라이언트**: 모델에 프롬프트를 보내는 데 사용합니다
- **MLflow**: 프롬프트를 관리하고 상호작용을 추적(trace)하는 데 사용합니다

직접 실습해 보겠습니다!

![](https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExNnRhZXg4eDlpZG83NnloeWJ6aGw3ZGp0bWl1azdobmNza2dobTNuYyZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/uENJtJp1SGd597E7tL/giphy.gif)

## 1. 모델에 연결하기

Canopy 개발자인 우리에게는 모델 접근 권한이 주어져 있습니다. 모델 서버는 **OpenAI 호환 API**를 제공하므로 표준 `openai` Python 라이브러리로 모델과 통신할 수 있습니다.

모델의 엔드포인트만 지정하면 됩니다. 

아래 상자를 클릭한 후에, 키보드에서 Shift + Enter 를 누르면 셸(명령어 상자) 단위로 명령어 실행 후, 다음 셸로 넘어갑니다. 

(Ctrl + Enter를 누르면 다음 셸로 넘어가지않고 이 셸에 머무릅니다)

In [ ]:
from openai import OpenAI

LLM_ENDPOINT = "http://llama-32-predictor.ai501.svc.cluster.local:8080"
MODEL_NAME = "llama32"

client = OpenAI(
    base_url=LLM_ENDPOINT + "/v1",
    api_key="no-key-required", #아직 API키가 필요없으니, 일단은 이 임시값을 설정해둡니다
)

## 2. 시스템 프롬프트 작성하기

앞에서 만든 시스템 프롬프트를 가져와 아래 셀을 수정하고 실험을 시작해 보세요. (예를 들어 'Use bullet Point.'라는 지시 사항을 넣으면 불렛 포인트를 사용해서 요약해줍니다) 

In [ ]:
sys_prompt = """You are a helpful assistant. Summarize this:"""

#### Canopy에 관한 사용자 메시지 예시입니다. 요약해 보겠습니다!

In [ ]:
user_message = """
캐노피(Canopy) — 생물학
생물학과 생태학에서 캐노피(Canopy, 수관층)는 숲이나 나무가 우거진 지역에서 나무의 수관(나무의 가지와 잎이 모여 이루는 부분)이 형성하는 상층부 또는 '지붕'을 의미합니다. 이 층은 숲 내부로 들어오는 햇빛의 양, 습도, 온도, 바람의 흐름을 조절함으로써 생태계의 균형을 유지하는 데 중요한 역할을 합니다. 캐노피는 일반적으로 가장 키가 큰 나무들의 가지와 잎으로 구성되며, 수 미터 두께에 달하는 빽빽하고 연속적인 덮개를 형성하기도 합니다.

캐노피의 주요 생태학적 기능 중 하나는 다양한 생물에게 서식지와 먹이를 제공하는 것입니다. 많은 조류, 곤충, 포유류, 착생식물(다른 식물에 붙어서 자라는 식물)은 이러한 높은 위치의 환경에서 살아가도록 특별히 적응해 왔습니다. 또한 캐노피는 폭우가 숲 바닥에 직접 미치는 충격을 줄이는 방어막 역할을 하여 토양 침식을 방지하고 토양의 비옥도를 유지하는 데 도움을 줍니다.

뿐만 아니라 캐노피는 햇빛을 흡수하고 이를 화학 에너지로 전환하는 광합성 과정에서 핵심적인 역할을 합니다. 이러한 에너지는 숲의 식물들이 생존하는 데 필요한 기반이 되며, 나아가 식물에 의존하는 동물들의 생존도 뒷받침합니다. 열대우림에서는 캐노피가 매우 빽빽하여 햇빛이 숲 바닥까지 거의 도달하지 못하는 경우가 많습니다. 이러한 환경은 숲의 하층 식생과 지표층에서 어떤 식물과 동물이 살아남을 수 있는지를 결정하는 중요한 요인이 됩니다.

과학자들은 캐노피의 구조, 생물다양성, 생태학적 기능을 더 잘 이해하기 위해 수관 관측용 크레인, 드론, 등반 장비와 같은 전문적인 도구와 연구 방법을 활용합니다. 이러한 연구를 통해 얻은 지식은 자연보호 활동에 매우 중요합니다. 특히 캐노피는 산림 벌채, 기후 변화, 인간 활동으로 인해 구조와 기능이 훼손되기 쉬우므로 더욱 세심한 보호가 필요합니다.

캐노피의 복잡한 구조를 이해하면 생태학자들은 숲속 생물들이 서로 의존하며 맺고 있는 관계와 산림 생태계가 제공하는 중요한 기능을 더욱 깊이 이해할 수 있습니다. 이러한 기능에는 탄소 저장, 산소 생산, 기후 조절 등이 포함됩니다. 따라서 캐노피를 보호하는 것은 생물다양성을 유지하고 지구 환경의 건강을 지키는 데 필수적입니다.

"""

## 3. 요청 보내기

스트리밍 chat completion 요청을 보내 보겠습니다. OpenAI를 사용할 때와 같은 형태의 API이며, 유일한 차이점은 `base_url`이 자체 호스팅 모델을 가리킨다는 것입니다.

In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_message}
    ],
    stream=True,
)

for chunk in response:
    if hasattr(chunk, 'choices') and len(chunk.choices) > 0:
        delta = chunk.choices[0].delta
        if hasattr(delta, 'content') and delta.content:
            print(delta.content, end="", flush=True)
print()

모델 서버가 OpenAI 프로토콜을 사용하므로 애플리케이션 코드를 바꾸지 않고도 모델을 교체할 수 있습니다. 엔드포인트와 모델 이름만 업데이트하면 됩니다. 모델별 로직은 필요하지 않습니다!

---
## 4. MLflow 사용하기

이제 **MLflow**를 활용해 보겠습니다. 현재 MLflow는 다음과 같은 일을 도와줍니다.

- 버전이 관리되는 레지스트리에서 **프롬프트를 관리**하여 어떤 프롬프트가 효과적이었는지 놓치지 않게 해 줍니다

먼저 MLflow 서버에 대한 연결을 설정하겠습니다.

In [ ]:
import os
import mlflow
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"

os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

NAMESPACE_PATH = "/run/secrets/kubernetes.io/serviceaccount/namespace"
if os.path.exists(NAMESPACE_PATH):
    with open(NAMESPACE_PATH) as f:
        os.environ["MLFLOW_WORKSPACE"] = f.read().strip()

SA_TOKEN_PATH = "/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("canopy-experiment")

print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Workspace: {os.environ.get('MLFLOW_WORKSPACE', 'not set')}")

## 5. 프롬프트 레지스트리

앞에서 MLflow Prompt Registry에 저장한 프롬프트를 기억하시나요? 여기에서 가져와 보겠습니다!

아래 `PROMPT_NAME`을 레지스트리에서 프롬프트를 만들 때 사용한 이름으로 수정하세요. 예를 들어 `summarization`이라는 이름을 사용했다면 그 이름을 입력합니다.

In [ ]:
PROMPT_NAME = "summarization"

In [ ]:
prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@latest")

print("Loaded prompt from registry:")
print(prompt.template)

In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": prompt.template},
        {"role": "user", "content": user_message}
    ],
    stream=True,
)

for chunk in response:
    if hasattr(chunk, 'choices') and len(chunk.choices) > 0:
        delta = chunk.choices[0].delta
        if hasattr(delta, 'content') and delta.content:
            print(delta.content, end="", flush=True)
print()

이제 프롬프트는 코드에 하드코딩되지 않고 레지스트리에 저장되어 있습니다. 프롬프트에 버전과 태그를 붙이고 여러 애플리케이션에서 공유할 수 있으며, 프롬프트를 업데이트할 때 코드를 바꿀 필요가 없습니다!

## 6. 새 프롬프트 버전 저장하기

프롬프트 버전은 코드로도 관리할 수 있습니다. 다른 템플릿으로 프롬프트의 새 버전을 등록하고 태그를 붙여 보겠습니다.

In [ ]:
new_prompt = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template="You are a concise academic assistant. Summarize the following text in 3 bullet points:",
    commit_message="Shorter bullet-point style summarization prompt",
    tags={"style": "bullet-points"},
)

mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="champion", version=new_prompt.version)

print(f"Registered prompt: {new_prompt.name}")
print(f"Version: {new_prompt.version}")
print(f"Alias: champion")
print(f"Template:\n{new_prompt.template}")

OpenShift AI 대시보드로 이동해 확인해 보세요. 태그와 `champion` 별칭(alias)이 붙은 새 버전의 프롬프트가 보일 것입니다. 같은 이름으로 `register_prompt`를 호출하면 새 버전이 자동으로 생성됩니다!

![mlflow-champion-prompt.png](./mlflow-champion-prompt.png)

이제 별칭으로 프롬프트를 불러와 사용해 보겠습니다.

In [ ]:
champion_prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@champion")

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": champion_prompt.template},
        {"role": "user", "content": user_message}
    ],
    stream=True,
)

for chunk in response:
    if hasattr(chunk, 'choices') and len(chunk.choices) > 0:
        delta = chunk.choices[0].delta
        if hasattr(delta, 'content') and delta.content:
            print(delta.content, end="", flush=True)
print()

---

실습 안내 문서로 돌아가 지금까지 다룬 내용을 Canopy 백엔드에 적용하고, LLM 로직을 프론트엔드에서 분리해 보세요!